# Monte Carlo and the CRLB: How Good Can an Estimator Be?

Every Q or frequency number in this library comes with an implicit question: how much of
the available information did the estimator actually use? This notebook answers it the
only way it can be answered cleanly — on synthetic data with known truth, comparing an
ensemble of estimates against the Cramér–Rao lower bound.

Two tools:

- **`CRLBCalculator`** — the theoretical floor on the variance of any unbiased
  estimator, computed from the explicit Fisher information of the ring-down model. No
  data required, only parameters.
- **`MonteCarloAnalyzer`** — repeats "generate a realization, run both estimators,
  record the error" many times, then reports the empirical spread.

**This is not the same thing as the plug-in uncertainty on real data.** The
`plugin_crlb_std_f` / `uncertainty_std_f` fields that
[`0.2_batch-analysis.ipynb`](0.2_batch-analysis.ipynb) puts into ratio tables are
computed from a *fitted* model on a *single real* record — no known truth, no ensemble.
Section 5 spells out the difference; it matters when reading either notebook's output.

**Prerequisites:** [`0.4_frequency-estimation.ipynb`](0.4_frequency-estimation.ipynb)
for the two estimators. No external data is needed. Runtime is under a minute.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ringdownanalysis import CRLBCalculator, MonteCarloAnalyzer, RingDownSignal, plots
from ringdownanalysis.plots import plot_performance_comparison, plot_q_performance_comparison

plots.apply_plotting_style()

## 1. The reference record

These are the parameters used throughout the technical note, `examples/usage_example.py`,
and the README "Key Results" section, so the numbers below are directly comparable to
them: $f_0 = 5$ Hz, $f_s = 100$ Hz, $N = 10^6$ (so $T = 10^4$ s), initial SNR 60 dB, and
$Q = 10^4$.

`RingDownSignal` derives everything else: the decay time $\tau = Q / (\pi f_0)$ and the
noise level $\sigma$ implied by the initial SNR, i.e. the SNR at $t = 0$ when the
amplitude is still $A_0$. Later in the record the local SNR is far lower, which is the
whole reason ring-down estimation differs from constant-amplitude tone estimation.

In [ ]:
f0 = 5.0  # Hz
fs = 100.0  # Hz
N = 1_000_000  # samples
A0 = 1.0
snr_db = 60.0  # initial SNR
Q_true = 10_000.0

signal = RingDownSignal(f0=f0, fs=fs, N=N, A0=A0, snr_db=snr_db, Q=Q_true)

print(f"tau      = {signal.tau:.2f} s")
print(f"T        = {signal.T:.0f} s  ->  T/tau = {signal.T / signal.tau:.2f}")
print(f"sigma    = {signal.sigma:.3e}  (initial SNR {snr_db:.0f} dB)")
print(f"amplitude at end of record: {A0 * np.exp(-signal.T / signal.tau):.3e}")
print(f"local SNR at end of record: {20 * np.log10(A0 * np.exp(-signal.T / signal.tau) / signal.sigma):.1f} dB")

## 2. The bound

For the ring-down model the Fisher information collapses into three weighted sums over
the decaying envelope,

$$
S_0 = \sum_n e^{-2t_n/\tau}, \qquad
S_1 = \sum_n t_n e^{-2t_n/\tau}, \qquad
S_2 = \sum_n t_n^2 e^{-2t_n/\tau},
$$

and the effective information about the angular frequency, after accounting for the
nuisance parameters $A_0$ and $\phi_0$, is
$I_{\text{eff}}(\omega) = (A_0^2/\sigma^2)\,\Delta S_2$ with
$\Delta S_2 = S_2 - S_1^2/S_0$. That gives

$$
\operatorname{Var}(\hat f) \ge \frac{1}{(2\pi)^2 I_{\text{eff}}(\omega)},
\qquad
\operatorname{Var}(\hat Q) \ge \frac{\sigma^2\tau^2}{4A_0^2\,\Delta S_2}\left(1 + 4Q^2\right).
$$

The $(1 + 4Q^2)$ factor is the reason high-Q resonators are hard: $Q = \pi f \tau$
amplifies any uncertainty in $\tau$, so the fractional precision on $Q$ degrades even
when the fractional precision on $f$ is excellent.

Two caveats worth carrying into section 4. The frequency bound is derived with $\tau$
treated as known, while the estimators fit $\tau$ jointly — so an efficiency slightly
below 1 is expected, not a bug. The Q bound additionally assumes the high-SNR,
many-cycle regime where $\omega$ and $\tau$ are asymptotically uncorrelated.

In [ ]:
crlb = CRLBCalculator()

crlb_var_f = crlb.variance(A0, signal.sigma, fs, N, signal.tau)
crlb_std_f = crlb.standard_deviation(A0, signal.sigma, fs, N, signal.tau)
crlb_var_q = crlb.q_variance(A0, signal.sigma, fs, N, signal.tau, f0)
crlb_std_q = crlb.q_standard_deviation(A0, signal.sigma, fs, N, signal.tau, f0)

print(f"Var(f) >= {crlb_var_f:.6e} Hz^2   ->  std >= {crlb_std_f:.6e} Hz")
print(f"                                       relative: {crlb_std_f / f0:.3e}")
print(f"Var(Q) >= {crlb_var_q:.6e}       ->  std >= {crlb_std_q:.6e}")
print(f"                                       relative: {crlb_std_q / Q_true:.3e}")

## 3. How the bound moves with record length

The bound is analytic, so the $T/\tau$ dependence costs nothing to map out. Holding
$\tau$, $\sigma$, and $f_s$ fixed and varying the number of samples shows two regimes:

- $T \ll \tau$: the decay barely matters, the record behaves like a constant-amplitude
  tone, and the bound falls as $T^{-3/2}$. The dashed reference line tracks the curve
  here and peels away as $T$ approaches $\tau$.
- $T \gtrsim$ a few $\tau$: the signal has decayed into the noise and extra samples add
  no information. The bound flattens at a floor set by $\tau$.

The practical consequence is that recording longer is only worth it up to a few decay
times; beyond that, improving $Q$ means improving $\tau$, the amplitude, or the noise.

In [ ]:
t_over_tau = np.logspace(-1.5, 1.5, 40)
n_grid = np.maximum(100, (t_over_tau * signal.tau * fs).astype(int))

std_f = np.array([crlb.standard_deviation(A0, signal.sigma, fs, int(n), signal.tau) for n in n_grid])
std_q = np.array([crlb.q_standard_deviation(A0, signal.sigma, fs, int(n), signal.tau, f0) for n in n_grid])
ratio = n_grid / fs / signal.tau

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

ax = axes[0]
ax.loglog(ratio, std_f, label="CRLB std(f)")
reference = std_f[0] * (ratio / ratio[0]) ** -1.5
ax.loglog(ratio, reference, "--", linewidth=1.2, label=r"$T^{-3/2}$ reference")
ax.axvline(1.0, color="C2", linestyle=":", linewidth=1.2, label=r"$T = \tau$")
ax.set_xlabel(r"$T/\tau$")
ax.set_ylabel("CRLB std(f) (Hz)")
ax.set_title("Frequency bound vs record length")
ax.set_ylim(std_f.min() / 3, std_f.max() * 3)
ax.legend()
ax.grid(True, which="both", alpha=0.3)

ax = axes[1]
ax.loglog(ratio, std_q / Q_true, color="C1", label="CRLB std(Q) / Q")
ax.axvline(1.0, color="C2", linestyle=":", linewidth=1.2, label=r"$T = \tau$")
ax.set_xlabel(r"$T/\tau$")
ax.set_ylabel("relative CRLB std(Q)")
ax.set_title("Relative Q bound vs record length")
ax.legend()
ax.grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()

floor_ratio = std_f[-1] / std_f[np.argmin(np.abs(ratio - 3.0))]
print(f"Going from T = 3 tau to T = {ratio[-1]:.0f} tau improves std(f) by only a factor {1 / floor_ratio:.2f}")

## 4. Monte Carlo: what the estimators actually achieve

`MonteCarloAnalyzer.run()` generates `n_mc` independent realizations of the same signal,
runs the NLS and DFT estimators on each, and returns the error arrays plus summary
statistics and the matching CRLB values. Trials are independent, so they run in parallel
across CPU cores.

`n_mc=50` keeps this interactive. That is a real limitation: the sampling uncertainty on
an estimated standard deviation is about $1/\sqrt{2(n_{mc}-1)}$, roughly 10% here, so
efficiencies below are good to one digit. The figures in the technical note use
`n_mc=100` at the same $N = 10^6$ (see `example_generate_latex_figures()` in
`examples/usage_example.py`).

In [ ]:
n_mc = 50

mc = MonteCarloAnalyzer()
results = mc.run(f0=f0, fs=fs, N=N, A0=A0, snr_db=snr_db, Q=Q_true, n_mc=n_mc, seed=42)

In [ ]:
rel_uncertainty = 1.0 / np.sqrt(2.0 * (n_mc - 1))
stats = results["stats"]

rows = []
for label, key, bound in [
    ("frequency (Hz)", "nls", results["crlb_std"]),
    ("frequency (Hz)", "dft", results["crlb_std"]),
    ("Q", "q_nls", results["crlb_std_q"]),
    ("Q", "q_dft", results["crlb_std_q"]),
]:
    rows.append(
        {
            "quantity": label,
            "estimator": "NLS" if key.endswith("nls") else "DFT",
            "bias": stats[key]["mean"],
            "bias stderr": stats[key]["std"] / np.sqrt(n_mc),
            "std": stats[key]["std"],
            "CRLB std": bound,
            "efficiency (CRLB/std)": bound / stats[key]["std"],
        }
    )

print(f"n_mc = {n_mc}: standard deviations carry about {100 * rel_uncertainty:.0f}% sampling uncertainty")
pd.DataFrame(rows).set_index(["quantity", "estimator"])

In [ ]:
axes_f = plot_performance_comparison(results)
fig_f = axes_f[0].figure if isinstance(axes_f, np.ndarray) else axes_f.figure
fig_f.set_size_inches(11, 4)
fig_f.suptitle("Frequency estimation", y=1.04)
plt.show()

axes_q = plot_q_performance_comparison(results)
fig_q = axes_q[0].figure if isinstance(axes_q, np.ndarray) else axes_q.figure
fig_q.set_size_inches(11, 4)
fig_q.suptitle("Q estimation", y=1.04)
plt.show()

## 5. Reading the result

**NLS is close to efficient.** Its frequency spread lands within a small factor of the
bound, and its bias is far below that spread. The remaining gap is expected: the bound
is computed with $\tau$ known, and the estimator pays for fitting $\tau$ as well. This
is the README "Key Results" claim, reproduced.

**DFT scatters several times wider on frequency**, because a Lorentzian fit to a handful
of periodogram bins discards information that the time-domain fit uses. It remains the
right tool when you need a fast, initialization-free frequency — see
[`0.4_frequency-estimation.ipynb`](0.4_frequency-estimation.ipynb) § 6.

**No estimator shows a significant bias here.** Compare each `bias` against its own
`bias stderr`: at `n_mc=50` every entry is within a couple of standard errors of zero,
so the errors are consistent with pure scatter. Detecting a genuine few-percent bias
would take hundreds of trials.

**Q spreads are essentially identical for both.** Neither estimator has a Q advantage:
`estimate_full()` obtains $\tau$ from a time-domain fit in both cases, and the frequency
route only enters through $Q = \pi f \tau$, where the $f$ error is negligible compared
to the $\tau$ error. Q accuracy is a $\tau$ problem, not a frequency problem.

### What this notebook does *not* tell you about real records

| | This notebook | `plugin_crlb_std_f` / `uncertainty_std_f` (used in [`0.2`](0.2_batch-analysis.ipynb)) |
| --- | --- | --- |
| Truth | known by construction | unknown |
| Statistic | spread over an ensemble of realizations | one number from one record |
| Inputs | true $A_0$, $\sigma$, $\tau$, $N$ | fitted $\tau$, residual noise, selected crop, with a residual-dof correction |
| Model assumptions | exactly satisfied by the synthetic data | violated by drift, plateau, and amplitude-dependent damping |
| Correct reading | how efficient the estimator is | a heuristic consistency diagnostic |

The README states the same caveat: batch ratios built from those plug-in values are
consistency diagnostics, not formal tests. On real ODIN and EDU records the dominant
error is not white noise at all — it is frequency drift and the ambient-driven plateau,
which no white-noise bound describes. That is why the pipeline prefers the
drift-immune `Q_demod` and reports a bootstrap CI for it instead of a CRLB.

### Next steps

- Where those real-record pathologies come from, on data:
  [`0.1_drifting-resonators.ipynb`](0.1_drifting-resonators.ipynb) and
  [`0.5_odin-phasemeter-data.ipynb`](0.5_odin-phasemeter-data.ipynb)
- Interval estimation when $\tau$ is poorly identified:
  [`0.3_profile-likelihood-q.ipynb`](0.3_profile-likelihood-q.ipynb)
- The estimator validated against pathological synthetics with known truth:
  [`20260819_EDU_SegmentedDemod_Estimator_Demo.ipynb`](20260819_EDU_SegmentedDemod_Estimator_Demo.ipynb)
- Full LaTeX figure set (`n_mc=100`, all four panels):
  `example_generate_latex_figures()` in `examples/usage_example.py`